# BERTopic: Linking Diffusion Depth to Topic Change
Phase 2 — Komponen 1

**Tujuan:** Mengukur apakah topic shift (dari narasi ekonomi/DPR ke kekerasan aparat)
berkorelasi dengan difusi yang lebih dalam/luas.

**Input:** `Data_bert_with_topics.csv` (output dari notebook BERTopic)

**Metrik diffusion depth:** reply_count, retweet_count, quote_count, favorite_count, view_count

## Cell 1: Import Library

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

print('Library berhasil di-import.')

## Cell 2: Load & Filter Data

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('Data_bert_with_topics.csv')
df['date'] = pd.to_datetime(df['date'])

# Filter: hanya tweet yang punya engagement data
df_eng = df[df['reply_count'].notna()].copy()

# Exclude outlier (topic -1)
df_analysis = df_eng[df_eng['topic'] != -1].copy()

print(f'Dataset asli: {len(df)} tweet')
print(f'Dengan engagement data: {len(df_eng)} tweet')
print(f'Setelah exclude outlier: {len(df_analysis)} tweet')
print(f'Topik tersedia: {df_analysis["topic"].nunique()}')

## Cell 3: Buat Composite Diffusion Score
Menggabungkan semua engagement metrics menjadi satu skor difusi.

In [ ]:
# Engagement columns
eng_cols = ['reply_count', 'retweet_count', 'quote_count', 'favorite_count', 'view_count']

# Isi NaN dengan 0 (untuk safety)
for col in eng_cols:
    df_analysis[col] = df_analysis[col].fillna(0)

# Log-transform karena distribusinya sangat skewed
for col in eng_cols:
    df_analysis[f'{col}_log'] = np.log1p(df_analysis[col])

# Composite diffusion score (rata-rata z-score dari semua engagement metrics)
log_cols = [f'{col}_log' for col in eng_cols]
for col in log_cols:
    mean = df_analysis[col].mean()
    std = df_analysis[col].std()
    df_analysis[f'{col}_z'] = (df_analysis[col] - mean) / std

z_cols = [f'{col}_z' for col in log_cols]
df_analysis['diffusion_score'] = df_analysis[z_cols].mean(axis=1)

print('Composite diffusion score berhasil dihitung.')
print(f'\nStatistik diffusion_score:')
print(df_analysis['diffusion_score'].describe())

## Cell 4: Diffusion Score per Topik
Perbandingan: topik mana yang difusinya paling dalam?

In [ ]:
# Rata-rata diffusion score per topik
topic_diffusion = df_analysis.groupby('topic').agg(
    count=('diffusion_score', 'size'),
    mean_diffusion=('diffusion_score', 'mean'),
    median_diffusion=('diffusion_score', 'median'),
    mean_retweet=('retweet_count', 'mean'),
    mean_favorite=('favorite_count', 'mean'),
    mean_reply=('reply_count', 'mean'),
    mean_view=('view_count', 'mean')
).sort_values('mean_diffusion', ascending=False)

# Tambahkan topic label
topic_labels = df_analysis.drop_duplicates('topic')[['topic', 'topic_label']].set_index('topic')
topic_diffusion = topic_diffusion.join(topic_labels)

print('=== Diffusion Score per Topik (tertinggi ke terendah) ===')
print(topic_diffusion[['topic_label', 'count', 'mean_diffusion', 'median_diffusion', 
                        'mean_retweet', 'mean_favorite']].to_string())

## Cell 5: Visualisasi — Diffusion Score per Topik

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

topic_diffusion_sorted = topic_diffusion.sort_values('mean_diffusion', ascending=True)

colors = []
for idx in topic_diffusion_sorted.index:
    label = topic_diffusion_sorted.loc[idx, 'topic_label']
    if any(kw in str(label).lower() for kw in ['affan', 'kurniawan']):
        colors.append('#e74c3c')  # merah — Affan
    elif any(kw in str(label).lower() for kw in ['polisi', '1312', 'cops', 'anjing']):
        colors.append('#e67e22')  # oranye — kekerasan aparat
    elif any(kw in str(label).lower() for kw in ['demo', 'dpr', 'prabowo', 'pajak', 'tuntutan', 'bubarkan', 'stopbayar']):
        colors.append('#3498db')  # biru — demo/ekonomi/DPR
    else:
        colors.append('#95a5a6')  # abu — lainnya

ax.barh(range(len(topic_diffusion_sorted)), topic_diffusion_sorted['mean_diffusion'], color=colors)
ax.set_yticks(range(len(topic_diffusion_sorted)))
ax.set_yticklabels([f"Topic {idx}: {topic_diffusion_sorted.loc[idx, 'topic_label'][:40]}" 
                     for idx in topic_diffusion_sorted.index], fontsize=9)
ax.set_xlabel('Mean Diffusion Score')
ax.set_title('Diffusion Score per Topic\n(Merah=Affan, Oranye=Kekerasan Aparat, Biru=Demo/DPR, Abu=Lainnya)')
plt.tight_layout()
plt.show()

## Cell 6: Kelompokkan ke Narrative Cluster
Mengelompokkan 19 topik ke tema besar untuk analisis yang lebih clean.

In [ ]:
# Mapping topik ke narrative cluster berdasarkan hasil BERTopic
# SESUAIKAN mapping ini jika diperlukan

narrative_map = {
    0: 'Demo & DPR',           # demo_dpr_gedung_mahasiswa
    1: 'Gerakan/Hashtag',      # indonesiagelap_resetindonesia
    2: 'Politik & Tuntutan',   # prabowo_dpr_presiden_tuntutan
    3: 'Kekerasan Aparat',    # polisi_polisipembunuh
    4: 'Ekonomi Rakyat',      # rakyat_tuntutan_pajak_gaji
    5: 'Affan Kurniawan',     # affan_kurniawan_almarhum
    6: 'Kekerasan Aparat',    # cops_1312_bastard
    7: 'Kekerasan Aparat',    # 1312_pembunuh_forever
    8: 'Gerakan/Hashtag',     # bubarkandprsontoloyo
    9: 'Keamanan & Respons',  # stay_safe_jaga
    10: 'Demo & DPR',         # temen_demo
    11: 'Kekerasan Aparat',   # anjing_polisi_1312
    12: 'Demo & DPR',         # demo_1312_dpr_budaya
    13: 'Kekerasan Aparat',   # 1312_medis_mata_mobil
    14: 'Gerakan/Hashtag',    # adilijokowi_indonesiagelap
    15: 'Gerakan/Hashtag',    # stopbayarpajak_bubarkandpr
    16: 'Ekonomi Rakyat',     # 80_indonesia_umkm
    17: 'Keamanan & Respons', # bendera_takut_pasang
    18: 'Keamanan & Respons'  # damai_indonesiaku_provokasi
}

df_analysis['narrative'] = df_analysis['topic'].map(narrative_map)

print('=== Distribusi per Narrative Cluster ===')
print(df_analysis['narrative'].value_counts())

## Cell 7: Diffusion Score per Narrative Cluster

In [ ]:
narrative_diffusion = df_analysis.groupby('narrative').agg(
    count=('diffusion_score', 'size'),
    mean_diffusion=('diffusion_score', 'mean'),
    median_diffusion=('diffusion_score', 'median'),
    std_diffusion=('diffusion_score', 'std'),
    mean_retweet=('retweet_count', 'mean'),
    mean_favorite=('favorite_count', 'mean'),
    mean_view=('view_count', 'mean')
).sort_values('mean_diffusion', ascending=False)

print('=== Diffusion per Narrative Cluster ===')
print(narrative_diffusion.to_string())

# Visualisasi
fig, ax = plt.subplots(figsize=(10, 6))
color_map = {
    'Affan Kurniawan': '#e74c3c',
    'Kekerasan Aparat': '#e67e22',
    'Demo & DPR': '#3498db',
    'Politik & Tuntutan': '#2980b9',
    'Ekonomi Rakyat': '#1abc9c',
    'Gerakan/Hashtag': '#9b59b6',
    'Keamanan & Respons': '#95a5a6'
}

narrative_sorted = narrative_diffusion.sort_values('mean_diffusion', ascending=True)
bar_colors = [color_map.get(n, '#95a5a6') for n in narrative_sorted.index]

ax.barh(range(len(narrative_sorted)), narrative_sorted['mean_diffusion'], color=bar_colors)
ax.set_yticks(range(len(narrative_sorted)))
ax.set_yticklabels([f"{n} (n={narrative_sorted.loc[n, 'count']})" for n in narrative_sorted.index])
ax.set_xlabel('Mean Diffusion Score')
ax.set_title('Diffusion Score per Narrative Cluster')
plt.tight_layout()
plt.show()

## Cell 8: Statistical Test — Apakah Perbedaan Signifikan?
Kruskal-Wallis test (non-parametrik, karena data engagement sangat skewed)

In [ ]:
# Kruskal-Wallis: apakah diffusion score berbeda signifikan antar narrative cluster?
groups = [group['diffusion_score'].values for name, group in df_analysis.groupby('narrative')]
stat, p_value = stats.kruskal(*groups)

print(f'=== Kruskal-Wallis Test ===')
print(f'H-statistic: {stat:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'Signifikan (p < 0.05): {"YA" if p_value < 0.05 else "TIDAK"}')

# Pairwise: Mann-Whitney antara narrative clusters kunci
print(f'\n=== Pairwise Mann-Whitney U Tests (key comparisons) ===')
key_pairs = [
    ('Demo & DPR', 'Affan Kurniawan'),
    ('Demo & DPR', 'Kekerasan Aparat'),
    ('Ekonomi Rakyat', 'Affan Kurniawan'),
    ('Ekonomi Rakyat', 'Kekerasan Aparat'),
    ('Affan Kurniawan', 'Kekerasan Aparat'),
    ('Gerakan/Hashtag', 'Affan Kurniawan')
]

for n1, n2 in key_pairs:
    g1 = df_analysis[df_analysis['narrative'] == n1]['diffusion_score']
    g2 = df_analysis[df_analysis['narrative'] == n2]['diffusion_score']
    if len(g1) > 0 and len(g2) > 0:
        u_stat, u_p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        # Effect size (rank-biserial correlation)
        r = 1 - (2 * u_stat) / (len(g1) * len(g2))
        print(f'{n1} vs {n2}: U={u_stat:.0f}, p={u_p:.6f}, effect size r={r:.3f} {"***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"}')

## Cell 9: Temporal — Diffusion Score Over Time per Narrative
Apakah difusi meningkat setelah topic shift?

In [ ]:
# Resample per hari
df_analysis['date_day'] = df_analysis['date'].dt.date

key_narratives = ['Demo & DPR', 'Kekerasan Aparat', 'Affan Kurniawan', 'Ekonomi Rakyat']

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Volume per hari per narrative
for narrative in key_narratives:
    subset = df_analysis[df_analysis['narrative'] == narrative]
    daily_count = subset.groupby('date_day').size()
    axes[0].plot(daily_count.index, daily_count.values, label=narrative, linewidth=2)

axes[0].set_ylabel('Tweet Count')
axes[0].set_title('Volume per Narrative Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Mean diffusion score per hari per narrative
for narrative in key_narratives:
    subset = df_analysis[df_analysis['narrative'] == narrative]
    daily_diff = subset.groupby('date_day')['diffusion_score'].mean()
    # Hanya plot hari yang punya minimal 5 tweet
    daily_count = subset.groupby('date_day').size()
    valid_days = daily_count[daily_count >= 5].index
    daily_diff_filtered = daily_diff[daily_diff.index.isin(valid_days)]
    axes[1].plot(daily_diff_filtered.index, daily_diff_filtered.values, label=narrative, linewidth=2)

axes[1].set_ylabel('Mean Diffusion Score')
axes[1].set_xlabel('Date')
axes[1].set_title('Mean Diffusion Score per Narrative Over Time (min 5 tweets/day)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Cell 10: Pre vs Post Topic Shift Comparison
Membandingkan diffusion score sebelum dan sesudah titik topic shift.

In [ ]:
# Tentukan titik topic shift berdasarkan topics over time dari BERTopic
# Dari visualisasi sebelumnya: shift terjadi sekitar 27 Agustus 2025
# SESUAIKAN tanggal ini jika diperlukan
shift_date = pd.Timestamp('2025-08-27')

df_analysis['phase'] = df_analysis['date'].apply(
    lambda x: 'Pre-shift' if x < shift_date else 'Post-shift'
)

print('=== Volume per Phase ===')
print(df_analysis['phase'].value_counts())

print('\n=== Diffusion Score: Pre-shift vs Post-shift per Narrative ===')
phase_comparison = df_analysis.groupby(['narrative', 'phase']).agg(
    count=('diffusion_score', 'size'),
    mean_diffusion=('diffusion_score', 'mean'),
    median_diffusion=('diffusion_score', 'median')
).round(4)

print(phase_comparison.to_string())

# Statistical test: pre vs post untuk setiap narrative
print('\n=== Mann-Whitney: Pre vs Post per Narrative ===')
for narrative in key_narratives:
    pre = df_analysis[(df_analysis['narrative'] == narrative) & (df_analysis['phase'] == 'Pre-shift')]['diffusion_score']
    post = df_analysis[(df_analysis['narrative'] == narrative) & (df_analysis['phase'] == 'Post-shift')]['diffusion_score']
    if len(pre) >= 5 and len(post) >= 5:
        u_stat, u_p = stats.mannwhitneyu(pre, post, alternative='two-sided')
        r = 1 - (2 * u_stat) / (len(pre) * len(post))
        print(f'{narrative}: pre(n={len(pre)}) vs post(n={len(post)}), U={u_stat:.0f}, p={u_p:.6f}, r={r:.3f} {"***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"}')
    else:
        print(f'{narrative}: insufficient data (pre={len(pre)}, post={len(post)})')

## Cell 11: Export Hasil

In [ ]:
# Simpan dataframe dengan diffusion score dan narrative labels
df_analysis.to_csv('Data_diffusion_analysis.csv', index=False)

from google.colab import files
files.download('Data_diffusion_analysis.csv')

print('File Data_diffusion_analysis.csv berhasil disimpan.')
print('File ini akan digunakan untuk Phase 2 Komponen 2: Social Penetration & Network Diffusion Framework.')